In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Keeps the notebook clean from pandas coercion warnings

preprocessing starts here 

In [10]:
def preprocess_sensor_data(df, is_train=True):
    # 1. Drop duplicates based on features (only if training)
    if is_train:
        cols_to_check = [c for c in df.columns if c != 'id']
        df = df.drop_duplicates(subset=cols_to_check).copy()
    else:
        df = df.copy()

    # 2. Identify 5,750 feature columns and coerce to numeric
    feature_cols = [c for c in df.columns if c not in ['id', 'target']]
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')
    
    # 3. Extract the 23 base sensor names 
    base_sensors = sorted(list(set(['_'.join(c.split('_')[1:]) for c in feature_cols if c.startswith('t')])))
    
    # 4. Interpolate and Clip Per Sensor
    for sensor in base_sensors:
        sensor_cols = [f"t{str(i).zfill(3)}_{sensor}" for i in range(250)]
        
        # a) Linear Interpolation across time steps
        df[sensor_cols] = df[sensor_cols].interpolate(method='linear', axis=1, limit_direction='both')
        
        # b) Outlier Clipping (1st & 99th Percentile)
        sensor_data = df[sensor_cols].values
        p01 = np.nanpercentile(sensor_data, 1)
        p99 = np.nanpercentile(sensor_data, 99)
        df[sensor_cols] = np.clip(df[sensor_cols], p01, p99)
        
        # c) Global median fallback for entirely NaN rows
        if df[sensor_cols].isna().any().any():
            sensor_median = np.nanmedian(sensor_data)
            df[sensor_cols] = df[sensor_cols].fillna(sensor_median)
            
    return df

In [11]:
from pathlib import Path

TRAIN_DATA_PATH = Path('data/train_data_iml.csv')
raw_train = pd.read_csv(TRAIN_DATA_PATH, low_memory=False)
processed_train = preprocess_sensor_data(raw_train, is_train=True)

# Keep identifiers separate and use sensor measurements as model inputs.
X_train = processed_train.drop(columns=['id', 'target'])
y_train = processed_train['target'].copy()
train_ids = processed_train['id'].copy()

print(f'Raw training data: {raw_train.shape}')
print(f'Processed training data: {processed_train.shape}')
print(f'X_train: {X_train.shape}; y_train: {y_train.shape}')
print(f'Remaining missing feature values: {int(X_train.isna().sum().sum())}')
display(X_train.head())
display(y_train.value_counts().sort_index())

Raw training data: (679, 5752)
Processed training data: (635, 5752)
X_train: (635, 5750); y_train: (635,)
Remaining missing feature values: 0


,t000_chest_acc_x,t000_chest_acc_y,t000_chest_acc_z,t000_chest_ecg_1,t000_chest_ecg_2,t000_ankle_acc_x,t000_ankle_acc_y,t000_ankle_acc_z,t000_ankle_gyro_x,t000_ankle_gyro_y,...,t249_ankle_mag_z,t249_arm_acc_x,t249_arm_acc_y,t249_arm_acc_z,t249_arm_gyro_x,t249_arm_gyro_y,t249_arm_gyro_z,t249_arm_mag_x,t249_arm_mag_y,t249_arm_mag_z
0,-8.8814,4.038000,-5.89440,0.410260,0.246990,1.27010,-10.455000,-3.59250,0.395180,-0.85366,...,-1.049000,3.0706,-7.27590,5.758400,-5.720403,-0.31828,-4.631537,-3.7619,0.8287,-11.285000
1,-4.1294,-0.095923,0.41242,9999.000000,-0.385140,0.20474,-7.038200,-0.84986,0.313540,-0.64916,...,-8.814200,-2.8387,-0.65825,45.544463,-0.739220,-0.52567,0.618530,3.5193,10.8200,-39.749001
2,-9.8024,-1.295100,-1.50600,-0.058608,-0.016745,0.97938,-60.142554,1.63560,0.328390,-0.62289,...,-0.593840,-1.6627,-9.08210,2.785500,-0.080392,-0.64066,0.971980,9999.0000,1.4479,-0.743390
3,3.1004,-0.564340,-0.07083,-0.481420,0.736790,9.57120,-3.551200,-5.61310,4.262806,-0.62664,...,18.962999,-1.9577,9.65440,2.756100,-1.029400,-0.15195,0.189660,-26.2320,-15.8490,34.535999
4,-6.8698,-0.422110,-1.08990,0.113030,1.209800,2.21730,-8.864500,2.66230,-0.233770,-0.89869,...,8.838100,-1.9680,-66.33279,0.342840,0.315690,-0.94867,-0.476290,-6.2384,4.5227,46.761002


target
arm_elevation      54
climbing_stairs    54
cycling            59
jogging            59
jump_front_back    17
knees_bending      50
lying              60
running            58
sitting            58
standing           58
waist_bends        49
walking            59
Name: count, dtype: int64

feature extraction

In [12]:
import numpy as np
import pandas as pd

# 1. Define the exact 23 channel names in alphabetical order
channels = [
    'ankle_acc_x', 'ankle_acc_y', 'ankle_acc_z',
    'ankle_gyro_x', 'ankle_gyro_y', 'ankle_gyro_z',
    'ankle_mag_x', 'ankle_mag_y', 'ankle_mag_z',
    'arm_acc_x', 'arm_acc_y', 'arm_acc_z',
    'arm_gyro_x', 'arm_gyro_y', 'arm_gyro_z',
    'arm_mag_x', 'arm_mag_y', 'arm_mag_z',
    'chest_acc_x', 'chest_acc_y', 'chest_acc_z',
    'chest_ecg_1', 'chest_ecg_2'
]

def reshape_2d_to_3d(df, channels):
    """
    Reshapes a 2D DataFrame of shape (N, 5750) into a 3D numpy array (N, 250, 23)
    """
    feature_cols = [c for c in df.columns if c not in ['id', 'target']]
    n_samples = len(df)
    
    # Re-order columns chronologically t000..t249 for each channel
    ordered_cols = []
    for t in range(250):
        t_str = f"t{str(t).zfill(3)}"
        for ch in channels:
            ordered_cols.append(f"{t_str}_{ch}")
            
    # Reshape array to (N, 250, 23)
    X_3d = df[ordered_cols].values.reshape(n_samples, 250, len(channels))
    return X_3d

def extract_features(X_3d):
    """
    Takes a 3D array of shape (Rows, 250 time steps, 23 sensors)
    and returns a 2D array of 115 extracted features.
    """
    means = np.mean(X_3d, axis=1)
    stds = np.std(X_3d, axis=1)
    mins = np.min(X_3d, axis=1)
    maxs = np.max(X_3d, axis=1)
    medians = np.median(X_3d, axis=1)
    
    # Combine horizontally: 23 sensors * 5 stats = 115 features per row
    features = np.hstack([means, stds, mins, maxs, medians])
    return features

# 2. Reshape cleaned data to 3D and extract features
X_3d = reshape_2d_to_3d(processed_train, channels)
X_features = extract_features(X_3d)

# 3. Create DataFrame with meaningful column names
feature_names = []
stat_names = ['mean', 'std', 'min', 'max', 'median']
for stat in stat_names:
    for ch in channels: 
        feature_names.append(f"{ch}_{stat}")

X = pd.DataFrame(X_features, columns=feature_names)

print("3D Matrix shape:", X_3d.shape)
print("Extracted Features DataFrame shape:", X.shape)

3D Matrix shape: (635, 250, 23)
Extracted Features DataFrame shape: (635, 115)


encoding labels


In [13]:
from sklearn.preprocessing import LabelEncoder

# 1. Encode the target labels into integers
le = LabelEncoder()
y = le.fit_transform(processed_train['target'])

# X was created by the 3D feature-extraction cell above.
print(f"Encoded labels shape: {y.shape}")
print(f"Feature matrix from 3D extraction: {X.shape}")

Encoded labels shape: (635,)
Feature matrix from 3D extraction: (635, 115)


train/validation/split


In [14]:
from sklearn.model_selection import train_test_split

# Create an 80/20 Train and Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Crucial for balancing the minority classes
)

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")

Training features shape: (508, 115)
Validation features shape: (127, 115)
